## Serialize a MultiDimensional Data using SchemaOrg and the CUAHSI.org-Raster extension

The purpose of this notebook is to evaluate how multidimensional data can be extracted and mapped to our Pydantic classes.

In [ ]:
!pip install -q netcdf4 zarr fsspec gcsfs rioxarray codetiming

In [ ]:
import os
import sys
import xarray
import base64
import hashlib
import rioxarray
import mimetypes
import numpy as np
from glob import glob
from pyproj import CRS
from pathlib import Path
from codetiming import Timer
from google.cloud import storage

# add the parent directory to the path. This is the 
# directory that contains our pydantic classes.
sys.path.append('..')
import base
import core
import dataset
import datavariable
import multidimensional

|Component|What it Describes|Example|Notes|
|---|---|---|--|
|Dimension	|Size/extent of an axis	|time = 12, lat = 90	|Abstract axis
|Variable	|Main data arrays	|temp(time, lat, lon)|	Can be scalar or multi-dimensional
|Coordinate Variable|	Values along a dimension|	lat(lat), time(time)|	Same name as dimension


### Helper Functions

In [ ]:
# Add raster MIME types if not already present
# These types were collected from: https://pystac.readthedocs.io/en/stable/api/media_type.html 
mimetypes.add_type("application/netcdf", ".nc")
mimetypes.add_type("application/vnd+zarr", ".zarr")

In [ ]:
def inspect_dimensions(ds: xarray.Dataset) -> None:
    # get dimension information
    print(f"{25*'-'}\nDimension Information\n{25*'-'}\n")
    for dimname, size in ds.sizes.items():
        print(f'* {dimname} --> Size: {size}\n')

In [ ]:
def inspect_coordinates(ds: xarray.Dataset) -> None:
    # get coordinate information
    print(f"{25*'-'}\nCoordinate Information\n{25*'-'}\n")
    for coordname in ds.coords.keys():
        print(f"* {coordname}")
        print(f"\tUnit: {ds.coords[coordname].attrs.get('units', 'unknown')}")
        print(f"\tDescription: {ds.coords[coordname].attrs.get('long_name', 'None')}")
        print(f"\tType: {ds.coords[coordname].dtype}")
        print(f"\tShape: {ds.coords[coordname].shape}")

In [ ]:
def inspect_variables(ds: xarray.Dataset) -> None:
    # get variable information
    # get coordinate information
    print(f"{25*'-'}\nVariable Information\n{25*'-'}\n")
    for varname in ds.variables.keys():
        if varname in ds.coords.keys():
            continue
            
        print(f"* {varname}")
        print(f"\tUnit: {ds.variables[varname].attrs.get('units', 'unknown')}")
        print(f"\tDescription: {ds.variables[varname].attrs.get('long_name', 'none')}")
        print(f"\tType: {ds.variables[varname].dtype}")
        print(f"\tCoordinates: ({', '.join(ds[varname].coords.keys())})")
        print(f"\tShape: {ds.variables[varname].shape}")

In [ ]:
def compute_sha256(file_path: Path) -> str:
    """
    Computes the SHA256 hash of a file.
    """
    sha256_hash = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            sha256_hash.update(chunk)
    return sha256_hash.hexdigest()


def hash_gcs_store(bucket_name:str, prefix:str) -> str:
    """
    This is a hack to convert a cloud bucket md5 into a sha256. This will not
    create a true sha256 of the object, but rather a sha256 of the md5sum.
    """
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blobs = bucket.list_blobs(prefix=prefix)
    
    sha256 = hashlib.sha256()

    for blob in sorted(blobs, key=lambda b: b.name):
        md5_bytes = base64.b64decode(blob.md5_hash)
        sha256.update(md5_bytes)
        
    return sha256.hexdigest()

In [ ]:
def get_crs_from_dataset_metadata(ds: xarray.Dataset) -> xarray.Dataset:
    """
    Set the CRS of an xarray.Dataset using metadata from a 'crs' or 'spatial_ref' variable.

    Parameters:
        ds (xarray.Dataset): Input dataset

    Returns:
        xarray.Dataset: Dataset with CRS set via rioxarray
    """
    
    # Identify CRS variable
    crs_var = None
    for name in ['crs', 'spatial_ref']:
        if name in ds.variables:
            crs_var = ds[name]
            break

    if crs_var is None:
        return None
        #raise ValueError("No CRS variable found (expected 'crs' or 'spatial_ref').")

    attrs = crs_var.attrs

    # Try to extract CRS from known attributes
    if "epsg_code" in attrs:
        crs = CRS.from_epsg(int(str(attrs["epsg_code"]).split(":")[-1]))
    elif "crs_wkt" in attrs:
        crs = CRS.from_wkt(attrs["crs_wkt"])
    elif "proj4_params" in attrs:
        crs = CRS.from_proj4(attrs["proj4_params"])
    elif "grid_mapping_name" in attrs and attrs["grid_mapping_name"] == "latitude_longitude":
        crs = CRS.from_epsg(4326)
    elif "esri_pe_string" in attrs:
        crs = CRS.from_wkt(attrs['esri_pe_string'])
    else:
        return None

    return crs

In [ ]:
def get_spatial_bounds(ds: xarray.Dataset) -> dict[str, float]:
    
    def is_lat(coord):
        std = coord.attrs.get("standard_name", "").lower()
        units = coord.attrs.get("units", "").lower()
        axis = coord.attrs.get("axis", "").upper()
        name = coord.name.lower()
        return (
            std == "latitude" or
            "degrees_north" in units or
            name in ["lat", "latitude", "y"] or
            axis == "Y"
        )

    def is_lon(coord):
        std = coord.attrs.get("standard_name", "").lower()
        units = coord.attrs.get("units", "").lower()
        axis = coord.attrs.get("axis", "").upper()
        name = coord.name.lower()
        return (
            std == "longitude" or
            "degrees_east" in units or
            name in ["lon", "longitude", "x"] or
            axis == "X"
        )

    lat_coord = None
    lon_coord = None

    for coord in ds.coords.values():
        if lat_coord is None and is_lat(coord):
            lat_coord = coord
        if lon_coord is None and is_lon(coord):
            lon_coord = coord

    if lat_coord is None or lon_coord is None:
        raise ValueError("Could not identify spatial coordinates.")

    # Handle 1D and 2D coordinate cases
    lat_vals = lat_coord.values
    lon_vals = lon_coord.values

    bounds = {
        "lat_min": float(np.nanmin(lat_vals)),
        "lat_max": float(np.nanmax(lat_vals)),
        "lon_min": float(np.nanmin(lon_vals)),
        "lon_max": float(np.nanmax(lon_vals)),
    }

    return bounds

### SchemaOrg Functions

In [ ]:
def build_dimensions(ds: xarray.Dataset) -> dict[datavariable.Dimension]:
    dims = {}
    for dimname, size in ds.sizes.items():
        var = ds.variables.get(dimname, None)
        attrs = var.attrs if var is not None else {}
    
        description = attrs.get('long_name', None)
        units = attrs.get('units', None)
        resolution = attrs.get('resolution', None)
        
        dims[dimname] = datavariable.Dimension(name=dimname,
                                           description=description,
                                           units=units,
                                           resolution=resolution,
                                           shape=ds.sizes.get(dimname))
    return dims

In [ ]:
def build_variables(ds: xarray.Dataset,
                    dims: dict[datavariable.Dimension],
                    compute_statistics=True) -> list[datavariable.DataVariable]:
    
    variables = []
    for varname in ds.variables.keys():
    
        v = ds[varname]
    
        # skip if this is a dimension
        if varname in v.dims:
            continue
            
        var_dims = [dims[d] for d in v.dims]

        # making this optional because it could be prohibitive
        # for large files.
        minValue = None
        maxValue = None
        if compute_statistics:
            minValue=v.min().item() or None
            maxValue=v.max().item() or None
    
        variables.append(datavariable.DataVariable(name = varname,
                         description = v.attrs.get('long_name', None),
                         unit = v.attrs.get('units', None),
                         dimension=var_dims or None,
                         shape=list(v.shape) or None,
                         minValue=minValue,
                         maxValue=maxValue,
                         dataType=str(v.dtype) or None,
                                    )
                        )
    return variables

In [ ]:
def build_coordinates(ds: xarray.Dataset,
                      dims: dict[datavariable.Dimension]) -> list[datavariable.DataVariable]:
    coords = []
    for coord_name, coord in ds.coords.items():
    
        coordinate_dimensions = [dims[dim_name] for dim_name in coord.dims]
        
        coords.append(datavariable.DataVariable(name = coord_name,
                                                description = coord.attrs.get('long_name', None),
                                                unit = coord.attrs.get('units', None),
                                                resolution = coord.attrs.get('resolution', None),
                                                dimension=coordinate_dimensions,
                                               )
                     )
    return coords

In [ ]:
def encode_netcdf(filepath: str,
                  validate_bbox:bool = True,
                  compute_statistics:bool = True) -> multidimensional.MultiDimensional:

    ds = xarray.load_dataset(filepath, engine='netcdf4')
    name = filepath.split('/')[1:][0]
    sha256 = compute_sha256(Path(filepath))
    file_url = 'https://'+filepath # hack to correct file_url format
    contentSize = f'{os.path.getsize(Path(filepath))/1024} KB'
    encodingFormat = mimetypes.guess_type(Path(filepath))[0]

    return encode_multidimensional_metadata(ds, file_url, name, sha256, contentSize, encodingFormat, validate_bbox, compute_statistics)
    
def encode_zarr(gcs_bucket_name: str,
                gcs_prefix: str, 
                validate_bbox:bool = True,
                compute_statistics:bool = True) -> multidimensional.MultiDimensional:


    with Timer(text="Loading Zarr... [{seconds:.3f} sec]"):
        zarr_url = f'gs://{gcs_bucket_name}/{gcs_prefix}'
        ds = xarray.open_zarr(zarr_url, consolidated=False)#, chunks={"time":-1, "lat":"auto", "lon":"auto"})

    with Timer(text="Computing Sha256... [{seconds:.3f} sec]"):
        sha256 = hash_gcs_store(gcs_bucket_name, gcs_prefix)
    
    name = gcs_prefix
    file_url = zarr_url
    contentSize = f'{ds.nbytes/1e9:.1f} GB'
    encodingFormat = mimetypes.guess_type(gcs_prefix)[0]
    
    return encode_multidimensional_metadata(ds, file_url, name, sha256, contentSize, encodingFormat, validate_bbox, compute_statistics)
    


def encode_multidimensional_metadata(ds: xarray.Dataset,
                                     file_url: str,
                                     file_name: str,
                                     sha256:str = None,
                                     contentSize:str = None, 
                                     encodingFormat:str = None, 
                                     validate_bbox:bool = True,
                                     compute_statistics:bool = True ) -> multidimensional.MultiDimensional:
    
    with Timer(text="Building SchemaOrg - Place... [{seconds:.3f} sec]"):
        bounds = get_spatial_bounds(ds)
        box_str = f"{bounds['lat_min']} {bounds['lon_min']} {bounds['lat_max']} {bounds['lon_max']}"
        geo = base.GeoShape(box = box_str, validate_bbox=validate_bbox)
        crs = get_crs_from_dataset_metadata(ds)
    
        if crs:
            srs = base.SpatialReference(
                name = crs.name,
                srsType=crs.type_name.split(' ')[0],
                code=crs.to_string(),
                wktString=crs.to_wkt()
            )
        else:
            srs = None
        
        place = base.Place(
            geo=geo,
            srs=srs
        )


    with Timer(text="Building SchemaOrg - Associated Media... [{seconds:.3f} sec]"):
        files = [
            base.MediaObject(contentUrl = file_url,
                             name = file_name,
                             sha256 = sha256,
                             contentSize = contentSize,
                             encodingFormat = encodingFormat)
        ]

    
    with Timer(text="Building SchemaOrg.Multidimensional - Dimensions... [{seconds:.3f} sec]"):
        dims = build_dimensions(ds)

    with Timer(text="Building SchemaOrg.Multidimensional - Variables... [{seconds:.3f} sec]"):
        variables = build_variables(ds, dims, compute_statistics)

    with Timer(text="Building SchemaOrg.Multidimensional - Coordinates... [{seconds:.3f} sec]"):
        coordinates = build_coordinates(ds, dims)
    
    with Timer(text="Building SchemaOrg.Multidimensional... [{seconds:.0f} sec]"):
        meta = multidimensional.MultiDimensional(variableMeasured=variables,
                                                 coordinates=coordinates,
                                                 associatedMedia=files,
                                                 spatialCoverage=place,
                                                 dimensions=dims.values(),
                                                )
    

    return meta

### Inspect a NetCDF File 

In [ ]:
ds = xarray.load_dataset('data/201806011600.LDASOUT_DOMAIN1.nc')
inspect_dimensions(ds)
inspect_coordinates(ds)
inspect_variables(ds)

### Encode a Local NetCDF

In [ ]:
meta = encode_netcdf('data/201806011600.LDASOUT_DOMAIN1.nc', validate_bbox=False)
print(meta.model_dump_json(exclude_none=True, indent=4))

### Encode a Zarr on GCP

In [ ]:
meta = encode_zarr('cesm2', 'ivt.zarr', validate_bbox=False, compute_statistics=False)
print(meta.model_dump_json(exclude_none=True, indent=4))